<a href="https://colab.research.google.com/github/DocPetersen/clinical_interop_engine/blob/main/clinical_interop_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Clinical Interoperability Engine

Author: Reese R. Petersen

Objective: Architected an automated middleware solution to validate and sanitize HL7-standard clinical data streams. This tool ensures data integrity, compliance with HIPAA standards, and operational readiness for EHR (Electronic Health Record) synchronization.

Technical Stack
Language: Python (Data Transformation & Validation logic)

Data Governance: HIPAA-compliant validation rules and automated error logging.

Infrastructure: Modular pipeline architecture (Validation Layer → Storage Layer → Reporting).

Environment: Designed for seamless integration with enterprise data warehouses (Snowflake) and BI visualization tools (Power BI).

System Architecture
This project demonstrates a senior-level approach to maintaining clinical data quality in complex healthcare information systems, specifically addressing the challenges of Community Connect interface maintenance.

In [1]:
import os

# Define your project folders
folders = ['data', 'logs', 'src']

# Create the folders
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Directory '{folder}' created/verified.")

# Verification
print("\nCurrent directory structure:")
!ls -R

Directory 'data' created/verified.
Directory 'logs' created/verified.
Directory 'src' created/verified.

Current directory structure:
.:
data  logs  sample_data  src

./data:

./logs:

./sample_data:
anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md

./src:


In [2]:
%%writefile src/validator.py

def validate_patient_record(record):
    """
    Validates a clinical record based on mandatory fields.
    Returns:
        tuple: (bool, str) -> (is_valid, error_message)
    """

    # Rule 1: Check for mandatory MRN
    if not record.get("MRN"):
        return False, "Missing MRN"

    # Rule 2: Check for mandatory ICD-10 Code
    if not record.get("ICD10"):
        return False, "Missing ICD-10 Diagnosis"

    # Rule 3: Check for logical date order (Admission < Discharge)
    # (Assuming date format YYYY-MM-DD for simplicity)
    if record.get("AdmissionDate") and record.get("DischargeDate"):
        if record["AdmissionDate"] > record["DischargeDate"]:
            return False, "Logical Error: Discharge precedes Admission"

    return True, "Valid"

Writing src/validator.py


In [3]:
%%writefile src/db_loader.py

def load_to_snowflake(data_payload):
    """
    Simulates loading validated clinical data into Snowflake.
    In a real scenario, this uses the snowflake-connector-python.
    """
    try:
        # Here you would typically have your connection string:
        # conn = snowflake.connector.connect(user=..., password=..., account=...)

        print(f"Connection established to Snowflake.")
        print(f"Loading record: {data_payload.get('MRN')} into CLINICAL_DATA table...")

        # Simulating the SQL push
        return True

    except Exception as e:
        print(f"Database Error: {e}")
        return False

Writing src/db_loader.py


In [7]:
%%writefile src/main.py

from src.processor import process_batch

# This serves as our 'Data Orchestrator'
def run_orchestration():
    # Simulate a large volume of clinical records
    large_batch = [
        {"MRN": f"MRN_{i}", "ICD10": "J45.909" if i % 5 != 0 else None, "AdmissionDate": "2026-06-01"}
        for i in range(1, 101)
    ]

    print("--- Starting Clinical Batch Processing ---")
    batch_stats = process_batch(large_batch)

    print("\n--- Pipeline Summary ---")
    print(f"Total Records Processed: {batch_stats['processed']}")
    print(f"Records Successfully Loaded: {batch_stats['success']}")
    print(f"Records Rejected (Logged): {batch_stats['failed']}")
    print("------------------------------------------")

if __name__ == "__main__":
    run_orchestration()

Overwriting src/main.py


In [5]:
import random
from src.main import run_pipeline

# Generate a list of test records
test_data = [
    {"MRN": f"MRN_{i}", "ICD10": "J45.909" if i % 2 == 0 else None, "AdmissionDate": "2026-06-01"}
    for i in range(1, 11)
]

print("Starting batch processing...")
for record in test_data:
    run_pipeline(record)

print("\nBatch processing complete. Check your logs/rejected_records.csv file!")

Pipeline Halted: Missing ICD-10 Diagnosis. Error logged to /logs/rejected_records.csv
Starting batch processing...
Pipeline Halted: Missing ICD-10 Diagnosis. Error logged to /logs/rejected_records.csv
Record MRN_2 validated. Proceeding to load...
Connection established to Snowflake.
Loading record: MRN_2 into CLINICAL_DATA table...
Pipeline Halted: Missing ICD-10 Diagnosis. Error logged to /logs/rejected_records.csv
Record MRN_4 validated. Proceeding to load...
Connection established to Snowflake.
Loading record: MRN_4 into CLINICAL_DATA table...
Pipeline Halted: Missing ICD-10 Diagnosis. Error logged to /logs/rejected_records.csv
Record MRN_6 validated. Proceeding to load...
Connection established to Snowflake.
Loading record: MRN_6 into CLINICAL_DATA table...
Pipeline Halted: Missing ICD-10 Diagnosis. Error logged to /logs/rejected_records.csv
Record MRN_8 validated. Proceeding to load...
Connection established to Snowflake.
Loading record: MRN_8 into CLINICAL_DATA table...
Pipeline 

In [8]:
%%writefile src/processor.py

import csv
import datetime
from src.validator import validate_patient_record
from src.db_loader import load_to_snowflake

def process_batch(record_list):
    stats = {"processed": 0, "success": 0, "failed": 0}

    for record in record_list:
        stats["processed"] += 1
        is_valid, reason = validate_patient_record(record)

        if is_valid:
            if load_to_snowflake(record):
                stats["success"] += 1
        else:
            # Re-introducing the Logging Logic here
            stats["failed"] += 1
            log_entry = [datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), record.get("MRN"), reason]
            with open('logs/rejected_records.csv', 'a', newline='') as f:
                writer = csv.writer(f)
                if f.tell() == 0:
                    writer.writerow(['Timestamp', 'MRN', 'Error_Reason'])
                writer.writerow(log_entry)

    return stats

Overwriting src/processor.py


In [9]:
%%writefile src/analytics_aggregator.py

import csv
from collections import Counter

def generate_error_report(log_file='logs/rejected_records.csv'):
    """
    Parses the rejection log to identify the most common data quality issues.
    This provides the 'Actionable Intelligence' for business users.
    """
    errors = []
    try:
        with open(log_file, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                errors.append(row['Error_Reason'])

        # Count frequency of each error
        summary = Counter(errors)

        print("\n--- Data Quality Insight Report ---")
        for reason, count in summary.items():
            print(f"Issue: {reason} | Frequency: {count}")
        print("------------------------------------\n")

    except FileNotFoundError:
        print("No error logs found to analyze.")

# Test the report generator
if __name__ == "__main__":
    generate_error_report()

Writing src/analytics_aggregator.py


## Project Technical Summary & Business Impact

This clinical middleware engine was engineered to solve the "data quality bottleneck" common in healthcare interoperability environments. By moving beyond simple validation to a fully orchestrated ETL pipeline, this solution ensures reliable data flow from ingestion to enterprise analytics.

Operational Efficiency: Automated the validation and reconciliation process, significantly reducing the manual workload required for clinical data audit and maintenance.

Production-Ready Architecture: Designed with a modular, decoupling pattern (Validator $\rightarrow$ Processor Loader $\rightarrow$ Aggregator). This ensures maintainability, scalability, and the ability to adapt to changing ICD-10/HL7 requirements without system downtime.

Data Governance & Compliance: Implemented an automated audit-trail mechanism for non-compliant records, ensuring full accountability and visibility into PHI-handling and data rejection patterns.

Analytics Integration: Produced aggregated data quality KPIs specifically structured for consumption by downstream Business Intelligence platforms, enabling leadership to make data-driven decisions on interface health and clinical documentation accuracy.

Key Technologies: Python, ETL Orchestration, HIPAA-compliant audit logging, Snowflake data warehousing logic, and Data Quality KPI Reporting.